# Triton Kernel Demo (RTX 3050)

This notebook demonstrates **Triton kernels** in this repo:
- **Matmul**: tiled C = A @ B with autotuning
- **Conv2D**: 3×3 convolution with tiling
- **Softmax, LayerNorm, GELU**: elementwise / reduction kernels
- **QKV & MLP**: fused transformer building blocks

**Run from repo root.** Requires: PyTorch with CUDA, `triton` (or `triton-windows` on Windows).

In [ ]:
import sys
import time
from pathlib import Path
root = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
try:
    import triton
    print(f"Triton {triton.__version__}")
except ImportError:
    print("Triton not installed (pip install triton or triton-windows)")

## Algorithm: Triton matmul

Triton matmul uses **tiled blocks**: each program computes a BLOCK_M×BLOCK_N tile of C by iterating over K in steps of BLOCK_K. Uses `tl.dot` (maps to tensor cores on GPU). **Autotuning** picks BLOCK_M, BLOCK_N, BLOCK_K for given M, N, K.

## Example: Matmul usage and correctness

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA required")

from triton_kernels.matmul import matmul_triton

M, K, N = 1024, 1024, 1024
A = torch.randn(M, K, device="cuda", dtype=torch.float16)
B = torch.randn(K, N, device="cuda", dtype=torch.float16)
C_triton = matmul_triton(A, B)
C_ref = torch.matmul(A.float(), B.float()).half()
err = (C_triton - C_ref).abs().max().item()
print(f"Matmul Triton shape: {C_triton.shape}, max diff: {err}")

## Example: Conv2D and LayerNorm

In [ ]:
from triton_kernels.conv import conv2d_triton
from triton_kernels.layernorm import layernorm_triton

B, C_in, H, W = 4, 1, 28, 28
C_out = 32
x = torch.randn(B, C_in, H, W, device="cuda", dtype=torch.float16)
w = torch.randn(C_out, C_in, 3, 3, device="cuda", dtype=torch.float16)
b = torch.zeros(C_out, device="cuda", dtype=torch.float16)
y = conv2d_triton(x, w, b, use_autotune=True)
print(f"Conv2D Triton output shape: {y.shape}")

x_ln = torch.randn(64, 128, device="cuda", dtype=torch.float16)
weight = torch.ones(128, device="cuda", dtype=torch.float16)
bias = torch.zeros(128, device="cuda", dtype=torch.float16)
y_ln = layernorm_triton(x_ln, (128,), weight, bias)
print(f"LayerNorm Triton output shape: {y_ln.shape}")

## Benchmark: PyTorch vs Triton (matmul, conv)

In [ ]:
def run_bench(fn, warmup=5, repeat=30):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeat):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000 / repeat

sizes = [512, 1024, 2048]
matmul_results = []
for N in sizes:
    A = torch.randn(N, N, device="cuda", dtype=torch.float16)
    B = torch.randn(N, N, device="cuda", dtype=torch.float16)
    t_pt = run_bench(lambda: torch.matmul(A, B))
    gflops = 2*N**3/1e9/(t_pt/1000)
    matmul_results.append((N, "PyTorch", t_pt, gflops))
    t_tr = run_bench(lambda: matmul_triton(A, B))
    gflops_tr = 2*N**3/1e9/(t_tr/1000)
    matmul_results.append((N, "Triton", t_tr, gflops_tr))

print("Matmul (FP16):")
for N, impl, ms, gf in matmul_results:
    print(f"  {N}³ {impl}: {ms:.4f} ms, {gf:.1f} GFLOPS")

In [ ]:
import torch.nn as nn
B, C_in, H, W = 128, 1, 28, 28
C_out = 32
x = torch.randn(B, C_in, H, W, device="cuda", dtype=torch.float16)
conv = nn.Conv2d(C_in, C_out, 3, padding=0).cuda().half()
w, b = conv.weight, conv.bias
t_pt = run_bench(lambda: torch.nn.functional.conv2d(x, w, b, padding=0))
t_tr = run_bench(lambda: conv2d_triton(x, w, b))
print(f"Conv2D (B={B}): PyTorch {t_pt:.4f} ms, Triton {t_tr:.4f} ms ({t_pt/t_tr:.2f}x)")

## Visualize matmul performance

In [ ]:
import matplotlib.pyplot as plt

labels = [f"{s}³" for s in sizes]
x_pos = range(len(labels))
pt_times = [r[2] for r in matmul_results if r[1] == "PyTorch"]
tr_times = [r[2] for r in matmul_results if r[1] == "Triton"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
w = 0.35
ax1.bar([i - w/2 for i in x_pos], pt_times, w, label="PyTorch", color="#2ecc71")
ax1.bar([i + w/2 for i in x_pos], tr_times, w, label="Triton", color="#3498db")
ax1.set_xticks(x_pos)
ax1.set_xticklabels(labels)
ax1.set_ylabel("Latency (ms)")
ax1.set_title("Matmul latency")
ax1.legend()

pt_gf = [r[3] for r in matmul_results if r[1] == "PyTorch"]
tr_gf = [r[3] for r in matmul_results if r[1] == "Triton"]
ax2.bar([i - w/2 for i in x_pos], pt_gf, w, label="PyTorch", color="#2ecc71")
ax2.bar([i + w/2 for i in x_pos], tr_gf, w, label="Triton", color="#3498db")
ax2.set_xticks(x_pos)
ax2.set_xticklabels(labels)
ax2.set_ylabel("GFLOPS")
ax2.set_title("Matmul throughput")
ax2.legend()
plt.tight_layout()
plt.show()